In [1]:
import json
import random
from pathlib import Path
from collections import Counter

In [2]:
path=Path('../datasets/chains_qwen2b.json')

with path.open('r',encoding='utf-8') as file:
    dataset=json.load(file)

print(len(dataset))

7496


In [3]:
dataset[0]['chains'][0]

{'text': " Let $B$ be the number of blue marbles and $Y$ be the number of yellow marbles in the bag.\nWe are given the following information:\n\n1. The ratio of blue marbles to yellow marbles is initially $8:5$.\n   This can be written as:\n   $\\frac{B}{Y} = \\frac{8}{5}$\n\n2. After removing 12 blue marbles and adding 21 yellow marbles, the ratio becomes $1:3$.\n   This can be written as:\n   $\\frac{B-12}{Y+21} = \\frac{1}{3}$\n\nFrom the first equation, we can express $B$ in terms of $Y$:\n$B = \\frac{8}{5}Y$\n\nSubstitute $B = \\frac{8}{5}Y$ into the second equation:\n$\\frac{\\frac{8}{5}Y - 12}{Y + 21} = \\frac{1}{3}$\n\nTo eliminate the fraction, multiply both sides by $15(Y + 21)$:\n$3(\\frac{8}{5}Y - 12) = 15(Y + 21)$\n\nSimplify:\n$\\frac{24}{5}Y - 36 = 15Y + 315$\n\nMultiply every term by 5 to clear the fraction:\n$24Y - 180 = 75Y + 1575$\n\nRearrange the equation to solve for $Y$:\n$24Y - 75Y = 1575 + 180$\n$-51Y = 1755$\n$Y = -\\frac{1755}{51}$\n$Y = -34.41$\n\nSince $Y$ c

In [4]:
def is_usable(chain: dict)->bool:
    return chain['finished'] and chain['n_tokens']<=700

In [5]:
random.seed(0)

pairs=[]
agree_candidates=[]

for i in dataset:
    correct=[c for c in i['chains'] if c['ok'] and is_usable(c)]
    wrong=[c for c in i['chains'] if (not c['ok']) and is_usable(c)]

    if correct and wrong:
        right_chain=random.choice(correct)
        wrong_chain=random.choice(wrong)
        if random.random()<0.5:
            pair_type='correction'
            answers=[wrong_chain,right_chain]
        else:
            pair_type='keep'
            answers=[right_chain,wrong_chain]
    elif len(correct)>=2:
        pair_type='agree'
        answers=random.sample(correct,2)
    else:
        continue

    record={
        'id': i['id'],
        'level': i['level'],
        'type': pair_type,
        'problem': i['problem'],
        'gold': i['gold'],
        'answer_1': answers[0]['text'].strip(),
        'answer_2': answers[1]['text'].strip(),
        'pred_1': answers[0]['pred'],
        'pred_2': answers[1]['pred'],
        'other_correct': [c['text'].strip() for c in correct],
    }

    if pair_type=='agree':
        agree_candidates.append(record)
    else:
        pairs.append(record)

print(len(pairs),len(agree_candidates))

2311 2243


In [6]:
agree_pairs=random.sample(agree_candidates,1000)
pairs=pairs+agree_pairs
random.shuffle(pairs)

print(Counter(p['type'] for p in pairs))

Counter({'correction': 1173, 'keep': 1138, 'agree': 1000})


In [7]:
Counter(p['level'] for p in pairs)

Counter({'Level 3': 887,
         'Level 4': 793,
         'Level 2': 755,
         'Level 5': 563,
         'Level 1': 311,
         'Level ?': 2})

In [8]:
assert all(p['pred_1']!=None and p['pred_2']!=None for p in pairs)

In [9]:
pairs[0]

{'id': 'math-2021',
 'level': 'Level 5',
 'type': 'correction',
 'problem': 'Expand $(2z^2 + 5z - 6)(3z^3 - 2z + 1)$.',
 'gold': '6z^5+15z^4-22z^3-8z^2+17z-6',
 'answer_1': '\\begin{aligned} (2z^2 + 5z - 6)(3z^3 - 2z + 1) &= 2z^2(3z^3 - 2z + 1) + 5z(3z^3 - 2z + 1) - 6(3z^3 - 2z + 1) \\\\ &= 6z^5 - 4z^3 + 2z^2 + 15z^4 - 10z^2 + 5z - 18z^3 + 12z - 6 \\\\ &= 6z^5 + 15z^4 - 22z^3 - 10z^2 + 12z - 6 \\end{aligned} \\boxed{6z^5 + 15z^4 - 22z^3 - 10z^2 + 12z - 6}',
 'answer_2': 'First, we need to expand the product \\((2z^2 + 5z - 6)(3z^3 - 2z + 1)\\). We will use the distributive property (also known as the FOIL method for binomials or the distributive property for polynomials). \\[ (2z^2 + 5z - 6)(3z^3 - 2z + 1) \\] We distribute each term in the first polynomial to each term in the second polynomial: \\[ 2z^2 \\cdot 3z^3 + 2z^2 \\cdot (-2z) + 2z^2 \\cdot 1 + 5z \\cdot 3z^3 + 5z \\cdot (-2z) + 5z \\cdot 1 - 6 \\cdot 3z^3 - 6 \\cdot (-2z) - 6 \\cdot 1 \\] Now, we compute each term: \\[ 2z^2 \

In [10]:
savepath=Path('../datasets/review_pairs.json')

with savepath.open('w',encoding='utf-8') as file:
    json.dump(pairs,file,ensure_ascii=False,indent=2)

print("Saved:",savepath)

Saved: ../datasets/review_pairs.json
